# 8. Mediciones de laboratorio
Procesa las mediciones del prototipo (carpeta `../Datos`) y las contrasta con el modelo $H_c(s)$ evaluado con los valores **medidos** de los componentes. Reproduce las tablas y las figuras de la Sección 8 del informe:

- `lab_bode_medido.png`: magnitud y fase del filtro completo frente al modelo y la plantilla.
- `lab_etapas_medido.png`: magnitud de cada sección por separado.

Instrumental: osciloscopio GW Instek GDS-1022 (puntas ×1, acoplamiento DC), generador con salida de 50 Ω, fuente simétrica de ±15 V, $V_i$ ≈ 4 Vpp.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path

DATOS, FIG = Path('../Datos'), Path('../Figuras'); FIG.mkdir(exist_ok=True)
comp = pd.read_csv(DATOS/'componentes_medidos.csv', index_col='componente')
tot  = pd.read_csv(DATOS/'barrido_total.csv')
st1  = pd.read_csv(DATOS/'etapa1.csv')
st2  = pd.read_csv(DATOS/'etapa2.csv')
lin  = pd.read_csv(DATOS/'linealidad_500Hz.csv')
comp['desvio_pct'] = (comp['medido']/comp['nominal'] - 1)*100
comp[['nominal', 'medido', 'tolerancia_pct', 'desvio_pct']].round(3)

## Modelo con valores nominales y con valores medidos

In [ ]:
def H1(f, R3, C3):
    s = 2j*np.pi*np.asarray(f, dtype=float); return 1/(1 + s*R3*C3)
def H2(f, R1, R2, C1, C2):
    s = 2j*np.pi*np.asarray(f, dtype=float); return 1/(s**2*R1*R2*C1*C2 + s*C1*(R1+R2) + 1)
def Hc(f, p): return H1(f, p['R3'], p['C3'])*H2(f, p['R1'], p['R2'], p['C1'], p['C2'])
db = lambda h: 20*np.log10(np.abs(h))
def fase(h): return np.degrees(np.unwrap(np.angle(h)))

nom = comp['nominal'].to_dict(); med = comp['medido'].to_dict()
for nombre, p in (('nominal', nom), ('medido', med)):
    w01 = 1/(p['R3']*p['C3']); w02 = 1/np.sqrt(p['R1']*p['R2']*p['C1']*p['C2'])
    Q = np.sqrt(p['R1']*p['R2']*p['C1']*p['C2'])/(p['C1']*(p['R1']+p['R2']))
    fg = np.logspace(2, 4.5, 200000); g = db(Hc(fg, p))
    print(f'{nombre:8s}: w01={w01:.0f} rad/s ({w01/2/np.pi:.0f} Hz)  w02={w02:.0f} rad/s ({w02/2/np.pi:.0f} Hz)  Q={Q:.3f}  '
          f'A1k={-db(Hc(1e3, p)):.2f} dB  A4k={-db(Hc(4e3, p)):.2f} dB  f-3dB={fg[np.argmin(abs(g+3))]:.0f} Hz')

## Barrido del filtro completo

In [ ]:
fd = np.logspace(1, 5, 400000)
ph_med = fase(Hc(fd, med))
def tabla(df, vin, vout, Hmod):
    out = df.copy()
    out['G_dB'] = db(df[vout]/df[vin])
    out['fase_deg'] = -360*df['f_Hz']*df['dt_us']*1e-6
    out['G_modelo_dB'] = db(Hmod(df['f_Hz']))
    out['fase_modelo_deg'] = np.interp(df['f_Hz'], fd, fase(Hmod(fd)))
    return out.round(2)
T = tabla(tot, 'Vi_Vpp', 'Vo_Vpp', lambda f: Hc(f, med))
T.loc[T['f_Hz'] == 10000, 'fase_deg'] = np.nan     # dt > periodo: fase no confiable
T

In [ ]:
ref = db(4.04/4.00)                                  # nivel a 100 Hz
x = np.log10([1422, 2000]); y = db(np.array([2.96, 1.52])/4.08) - ref
f3 = 10**np.interp(-3, y[::-1], x[::-1])
print(f'f(-3 dB) medida = {f3:.0f} Hz = {2*np.pi*f3:.0f} rad/s  (ventana 7870 a 10012 rad/s)')
print(f'A(1 kHz) = {-db(3.88/4.08):.2f} dB (<= 1 dB)')
print(f'A(4 kHz) lectura directa = {-db(0.24/4.08):.2f} dB (>= 24 dB)')
print(f'A(4 kHz) producto de secciones = {-(db(1.40/4.08) + db(0.21/1.44)):.2f} dB')
print(f'Pendiente 2-4 kHz = {db(0.24/4.08) - db(1.52/4.08):.1f} dB/octava  (modelo {db(Hc(4e3, med)) - db(Hc(2e3, med)):.1f})')

In [ ]:
AZ, GR, NA = '#2a6fdb', '#888888', '#e8710a'
fg = np.logspace(np.log10(50), np.log10(20000), 3000)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 6.6), sharex=True)
a1.fill_between([50, 1000], -60, -1, color='#f2d7d5', lw=0)
a1.fill_between([4000, 20000], -24, 3, color='#f2d7d5', lw=0, label='Zona prohibida (plantilla)')
a1.semilogx(fg, db(Hc(fg, nom)), color=GR, lw=1.6, ls='--', label='Modelo, valores comerciales nominales')
a1.semilogx(fg, db(Hc(fg, med)), color=AZ, lw=2, label='Modelo, valores medidos de los componentes')
ok = T['f_Hz'] <= 4000
a1.semilogx(T['f_Hz'][ok], T['G_dB'][ok], 'o', ms=7, color=NA, mec='white', mew=1.2, label='Medición')
a1.semilogx(T['f_Hz'][~ok], T['G_dB'][~ok], 'o', ms=7, mfc='white', mec=NA, mew=1.5, label='Medición limitada por resolución')
a1.set_ylim(-60, 3); a1.set_ylabel('Ganancia [dB]')
a1.legend(loc='lower left', fontsize=8.5, frameon=True, facecolor='white', edgecolor='#cccccc', framealpha=1)
a1.set_title('Respuesta en frecuencia del filtro: medición frente a modelo', fontsize=11, loc='left')
a2.semilogx(fg, fase(Hc(fg, nom)), color=GR, lw=1.6, ls='--')
a2.semilogx(fg, fase(Hc(fg, med)), color=AZ, lw=2)
ok2 = T['fase_deg'].notna()
a2.semilogx(T['f_Hz'][ok2], T['fase_deg'][ok2], 'o', ms=7, color=NA, mec='white', mew=1.2)
a2.set_ylabel('Fase [°]'); a2.set_xlabel('Frecuencia [Hz]'); a2.set_ylim(-290, 10)
a2.set_yticks([0, -45, -90, -135, -180, -225, -270]); a2.set_xlim(50, 20000)
for a in (a1, a2): a.grid(True, color='#dddddd', lw=0.6)
fig.tight_layout(); fig.savefig(FIG/'lab_bode_medido.png', dpi=200); plt.show()

## Cada sección por separado: $f_{01}$, $f_{02}$ y $Q$

In [ ]:
T1 = tabla(st1, 'Vi_Vpp', 'Vo1_Vpp', lambda f: H1(f, med['R3'], med['C3']))
T2 = tabla(st2, 'Vo1_Vpp', 'Vo_Vpp', lambda f: H2(f, med['R1'], med['R2'], med['C1'], med['C2']))
T2.loc[T2['f_Hz'] == 8000, 'fase_deg'] = np.nan
display(T1); display(T2)
print(f'Q estimado (|H2| en 1422 Hz) = {2.96/2.92:.3f}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.4), sharey=True)
datos = ((axs[0], lambda f: H1(f, med['R3'], med['C3']), T1, 'Etapa 1: $V_{o1}/V_i$'),
         (axs[1], lambda f: H2(f, med['R1'], med['R2'], med['C1'], med['C2']), T2, 'Etapa 2: $V_o/V_{o1}$'))
for ax, H, d, tit in datos:
    ax.semilogx(fg, db(H(fg)), color=AZ, lw=2, label='Modelo (valores medidos)')
    ax.semilogx(d['f_Hz'], d['G_dB'], 'o', ms=7, color=NA, mec='white', mew=1.2, label='Medición')
    ax.set_title(tit, fontsize=10.5, loc='left'); ax.set_xlabel('Frecuencia [Hz]'); ax.set_xlim(100, 10000)
    ax.grid(True, color='#dddddd', lw=0.6)
axs[0].set_ylabel('Ganancia [dB]'); axs[0].set_ylim(-32, 4); axs[0].legend(fontsize=8.5, frameon=False, loc='lower left')
fig.tight_layout(); fig.savefig(FIG/'lab_etapas_medido.png', dpi=200); plt.show()

## Linealidad (500 Hz) y respuesta al escalón

In [ ]:
lin['G_dB'] = db(lin['Vo_Vpp']/lin['Vi_Vpp']).round(2)
print(f'Ganancia esperada en 500 Hz: {db(Hc(500, med)):.3f} dB'); display(lin)

R3, C3, R1, R2, C1, C2 = (med[k] for k in ('R3', 'C3', 'R1', 'R2', 'C1', 'C2'))
den = np.polymul([R3*C3, 1], [R1*R2*C1*C2, C1*(R1+R2), 1])
t = np.linspace(0, 3e-3, 30001); _, y = signal.step(([1], den), T=t)
i10, i90, ip = np.argmax(y >= 0.1), np.argmax(y >= 0.9), np.argmax(y)
print(f'Modelo: tr(10-90) = {(t[i90]-t[i10])*1e6:.0f} us, sobrepico = {(y.max()-1)*100:.2f} %, '
      f'pico a {(t[ip]-t[i10])*1e6:.0f} us del 10 %')
print('Medido: tr = 260 us, sobrepico ~ 6.5 %, pico a 440 us del 10 %')